In [ ]:
%reload_ext autoreload
#%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from tqdm import tqdm
import pylab as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import itertools
from g_anomaly import PA_with_anomaly_numba
from g_standard import standard_PA_numba
from support_tools import (
    edge_degree_counting,
    edge_degree_counting_many,
    expected_degree_PA_normal_all_vertices,
    graph_series_connected_vertex_degree,
    get_top_vertices,
    prepare_screening,
    get_vertex_ranks
)
from estimation_pa_anomaly_joint import iteration_estimation_update
from estimation_pa_anomaly_fixed import single_estimation_beta, single_estimation_delta
from estimation_pa_normal import optimize_parameters_normal
from pa_loglikelihood import neg_log_likelihood_normal, neg_log_likelihood_anomaly

In [ ]:
def plot_estimation_distribution(data_list, data_true, label, tau, label_2):
    
    data_mean = np.mean(data_list)
    data_max = np.max(data_list)
    data_min = np.min(data_list)
    data_std = np.std(data_list, ddof=1)  # ddof=1, sample standard deviation

    ci_lower = np.percentile(data_list, 5)
    ci_upper = np.percentile(data_list, 95)

    print(f"The mean of {label} estimation is {data_mean:4f}.")
    print(f"5% quantile: {ci_lower:.4f}, 95% quantile: {ci_upper:.4f}")
    print(f"The maximum, minimum, std for {label} are: {data_max:.4f}, {data_min:.4f}, {data_std:.6f}")

    # histogram
    sns.histplot(data_list, bins=30, kde=True, color="skyblue", edgecolor="black")

    plt.axvline(data_mean, color='red', linestyle='--',
                label=f"Mean: {data_mean:.4f}")
    plt.axvline(ci_lower, color='green', linestyle=':',
                label=f"5% quantile")
    plt.axvline(ci_upper, color='green', linestyle=':',
                label=f"95% quantile")

    plt.legend(loc="upper right")
    plt.ylabel(f"Density of {label}")
    plt.title(f"Estimation of {label} ({label_2})")

    """plt.figtext(
        0.5, -0.08,
        f"5% quantile = {ci_lower:.4f},   95% quantile = {ci_upper:.4f}",
        ha="center",
        fontsize=10
    )"""

    plt.tight_layout()
    plt.savefig(f"MLE_of_{label}_{tau}.png",
            dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
def run_single_dataset_estimate_joint(beta_init, delta_init, network_size, m, edge_list, k_result, vertex_list):
    """
    top_k_degree: choose the vertices with top k degree as vertex candidates
    top_k_ratio: choose the vertices with top k ratio as vertex candidates
    vertex_i: 0-based, but in the calculation, vertex index should be 1-based
    log_likelihood_anomaly_numba: return the negative log-likelihood value
    procedure: 
        1. load the file, find the degree_k at every time step, and the vertices that might be anomaly
        2. estimation: find several candidates, multi-start of beta, minimize negative log-likelihood function
        3. find the vertex that has maximum log likelihood value
    return:
        tau_hat, beta_hat, delta_hat, log-likelihood value
    """
    
    beta_list, delta_list, log_L_list= [], [], []
        
    #edge_list = np.load(file_name).tolist()
    edges_np = np.asarray(edge_list, dtype=np.int64)          # shape (n,2)
    verts_np = np.asarray(vertex_list, dtype=np.int64)        # 0-based candidates

    deg_all, recv_all = edge_degree_counting_many(verts_np, edges_np, network_size)

    for i in range(len(vertex_list)):
        vertex_i = vertex_list[i] + 1  # likelihood: 1-based
        degree_series_i = deg_all[i].astype(np.float64)   
        edge_adj_i      = recv_all[i].astype(np.float64) 

        beta_i, delta_i, log_L_i = iteration_estimation_update(
            network_size, vertex_i, k_result,
            degree_series_i, edge_adj_i, m,
            beta_init, delta_init
        )
        log_L_list.append(log_L_i)
        beta_list.append(beta_i)
        delta_list.append(delta_i)
        #print(beta_i, delta_i, log_L_i)
    vertex_best = np.argmin(log_L_list)
    tau_hat = vertex_list[vertex_best]
    """beta_hat = round(beta_list[vertex_best],4)
    delta_hat = round(delta_list[vertex_best],4)
    log_L_hat = round(log_L_list[vertex_best],4)"""
    beta_hat =beta_list[vertex_best]
    delta_hat = delta_list[vertex_best]
    log_L_hat = log_L_list[vertex_best]

    return tau_hat, beta_hat, delta_hat, log_L_hat

################ one parameter estimation #############
def run_single_dataset_estimate_fixed(edge_list, exp_degree, beta_bounds, delta_bounds, beta_fixed, delta_fixed, vertex_tau, network_size, m, top_k_degree, top_k_ratio):
    """
    vertex_i: 0-based, but in the calculation, vertex index should be 1-based
    log_likelihood_anomaly_numba: return the negative log-likelihood value
    procedure: 
        1. load the file, find the degree_k at every time step, and the vertices that might be anomaly
        2. estimation: fixed delta, find beta maximizing log likelihood function; use the best beta to find delta that maximize log likelihood function
        3. find the vertex that has maximum log likelihood value
    return:
        tau_hat, beta_hat, delta_hat, log-likelihood value
    """
    
    beta_list, delta_list, log_L_list= [], [], []

    k_series = graph_series_connected_vertex_degree(edge_list, receiver_index=1)
    k_series = np.asarray(k_series, dtype=np.float64)

    degree_series_tau, edge_adjacency_tau = edge_degree_counting(vertex_tau -1 , edge_list)
    degree_series_tau = np.asarray(degree_series_tau, dtype=np.float64)
    edge_adjacency_tau = np.asarray(edge_adjacency_tau, dtype=np.float64)
        
    vertex_list_tmp = get_top_vertices(edge_list, exp_degree, top_k_degree, top_k_ratio,0)
     
    beta_hat = single_estimation_beta(beta_bounds,delta_fixed, network_size, vertex_tau, k_series, degree_series_tau, edge_adjacency_tau,m)
    delta_hat = single_estimation_delta(delta_bounds,beta_fixed, network_size, vertex_tau, k_series, degree_series_tau, edge_adjacency_tau,m)
    ### fix beta and delta, we choose the  vertices with top k ratio or top k degree to find the one with maximal likelihood 
    for i in range(len(vertex_list_tmp)):
        vertex_i = vertex_list_tmp[i] + 1
        degree_series_i, edge_adjacency_i = edge_degree_counting(vertex_i -1 , edge_list)
        degree_series_i = np.asarray(degree_series_i, dtype=np.float64)
        edge_adjacency_i = np.asarray(edge_adjacency_i, dtype=np.float64)
            
        neg_log_L_i = neg_log_likelihood_anomaly(beta_fixed, delta_fixed, network_size, vertex_i,
                                k_series, degree_series_i, edge_adjacency_i, m)
        log_L_list.append(neg_log_L_i)
            #print(beta_i, delta_i, log_L_i)
    vertex_best = np.argmin(log_L_list)
    tau_hat = vertex_list_tmp[vertex_best]

    return tau_hat, beta_hat, delta_hat

In [ ]:
############### sample of individual estimation #############

delta_true = 0.1
beta_true = 0.5

network_size_tmp = 100000
m_tmp = 3
vertex_tau = 100

Top_degree_rank = 5
Top_ratio_rank = 5

beta_bound_tmp = [0.000001,10]
delta_bound_tmp = [-m_tmp, m_tmp]


exp_degree = expected_degree_PA_normal_all_vertices(network_size_tmp, delta_true, m_tmp)

best_records = []

for rep_1 in tqdm(range(300)):
    seed_1 = 123 + rep_1   # tau=100, seed=123; tau=50000,seed = 12345; tau=99000, seed = 98765

    edges_1 = PA_with_anomaly_numba(network_size_tmp, m_tmp, beta_true, vertex_tau, delta_true, seed_1)
    tau_hat_i, beta_hat_i, delta_hat_i = run_single_dataset_estimate_fixed(
            edges_1, exp_degree, beta_bound_tmp, delta_bound_tmp, beta_true, delta_true, vertex_tau, network_size_tmp, m_tmp,
                Top_degree_rank, Top_ratio_rank)

    rec = {
        "dataset": rep_1,
        "seed": seed_1,
        "m_parameter": m_tmp,
        "network_size": network_size_tmp,
        "beta_true": beta_true,
        "delta_true": delta_true,
        "tau_true": vertex_tau,
        "beta_hat": beta_hat_i,
        "delta_hat": delta_hat_i,
        "tau_hat": tau_hat_i,
        }

    best_records.append(rec)

best_df = pd.DataFrame(best_records)

beta_hat_tmp = best_df["beta_hat"].tolist()
plot_estimation_distribution(beta_hat_tmp, beta_true, "beta", vertex_tau, "Early anomaly")
delta_hat_tmp = best_df["delta_hat"].tolist()
plot_estimation_distribution(delta_hat_tmp, delta_true, "delta", vertex_tau, "Early anomaly")

In [ ]:
############### sample of joint estimation #############

beta_initials = [1.0,3.0, 5.0, 7.0, 9.0]
delta_initials = 0.001

delta_true = 0.1
beta_true = 0.5

network_size_tmp = 100000
m_tmp = 3
vertex_tau = 50000

Top_degree_rank = 5
Top_ratio_rank = 5

exp_degree_tmp = expected_degree_PA_normal_all_vertices(network_size_tmp, delta_true, m_tmp)
best_records = []

for rep_2 in tqdm(range(300)):
    seed_2 = 12345 + rep_2      # base_seed: tau=100, seed=98765; tau=50000,seed = 12345; tau=99000, seed = 8765
    edges_2 = PA_with_anomaly_numba(network_size_tmp, m_tmp, beta_true, vertex_tau, delta_true, seed_2)
    k_series_2 = np.asarray(graph_series_connected_vertex_degree(edges_2, receiver_index=1), dtype=np.float64)
    vertex_list_2 = get_top_vertices(edges_2, exp_degree_tmp, Top_degree_rank, Top_ratio_rank,delta_true)

    for b0 in beta_initials:
        tau_hat_i, beta_hat_i, delta_hat_i, log_L_hat_i = run_single_dataset_estimate_joint(
            b0, delta_initials, network_size_tmp, m_tmp, edges_2, k_series_2, vertex_list_2
        )

        rec = {
            "dataset": rep_2,
            "seed": seed_2,
            "beta_init": b0,
            "delta_init": delta_initials,
            "beta_hat": beta_hat_i,
            "delta_hat": delta_hat_i,
            "tau_hat": tau_hat_i,
            "nll": log_L_hat_i,
        }
        best_records.append(rec)

best_df = pd.DataFrame(best_records)



In [ ]:
### for final results with multi-start of beta, we need to choose the one with smallest nll value

idx = best_df.groupby("dataset")["nll"].idxmin()
best_per_dataset = best_df.loc[idx].reset_index(drop=True)
print(len(best_per_dataset))
print(best_per_dataset["dataset"].nunique())

beta_hat_list = best_per_dataset["beta_hat"].tolist()
delta_hat_list = best_per_dataset["delta_hat"].tolist()
best_per_dataset["tau_correct"] = (best_per_dataset["tau_hat"] == vertex_tau - 1)

summary = pd.DataFrame({
    "beta_mean":  [best_per_dataset["beta_hat"].mean()],
    "beta_std":   [best_per_dataset["beta_hat"].std()],
    "beta_q05":   [best_per_dataset["beta_hat"].quantile(0.05)],
    "beta_q95":   [best_per_dataset["beta_hat"].quantile(0.95)],

    "delta_mean": [best_per_dataset["delta_hat"].mean()],
    "delta_std":  [best_per_dataset["delta_hat"].std()],
    "delta_q05":  [best_per_dataset["delta_hat"].quantile(0.05)],
    "delta_q95":  [best_per_dataset["delta_hat"].quantile(0.95)],

    "tau_correct_rate": [best_per_dataset["tau_correct"].mean()]
})


summary = summary.round(4)
print(summary)
    
plot_estimation_distribution(beta_hat_list, beta_true, "beta", vertex_tau, "Midway anomaly")
plot_estimation_distribution(delta_hat_list, delta_true, "delta", vertex_tau, "Midway anomaly")

In [ ]:

###### the violin plot for different initial values ###########

fig, ax = plt.subplots(figsize=(7, 5))

groups = []
labels = []

for b0 in sorted(best_df["beta_init"].unique()):
    groups.append(best_df.loc[best_df["beta_init"] == b0, "beta_hat"].values)   # change beta_hat to delta_hat then we get the plot for delta estimates
    labels.append(f"{b0:.4f}")

parts = ax.violinplot(
    groups,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

for pc in parts["bodies"]:
    pc.set_facecolor("#f4a6b5")   # soft pink
    pc.set_edgecolor("black")
    pc.set_alpha(0.7)
# style
for pc in parts["bodies"]:
    pc.set_alpha(0.6)

ax.axhline(beta_true, linestyle="--", color="red", linewidth=1)

ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)

ax.set_xlabel(r"Initial value $\beta_0$")
ax.set_ylabel(r"Estimated $\hat{\beta}$")
ax.set_title("Distribution of $\\hat{\\beta}$ for different initial values")
plt.savefig(f"violin_plot_beta_{vertex_tau}.png")
plt.tight_layout()
plt.show()